In [16]:
%pip install numpy pandas matplotlib
%pip install langdetect
%pip install deep-translator
%pip install wordninja


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from langdetect import detect
from deep_translator import GoogleTranslator
import re
import wordninja


df = pd.read_csv('corporate_law_data.csv')
df.head()

,title,summary,content,links,url
0,Corporate law,Corporate law(also known ascompany laworenterp...,Anguilla\nAustralia\nBVI\nCanada\nCayman Islan...,"['/wiki/Chairman', '/wiki/Takeover_Code', '/wi...",https://en.wikipedia.org/wiki/Corporate_law
1,Corporation,NaN,NaN,['/wiki/Wikipedia:Protection_policy#pending'],https://en.wikipedia.org/wiki/Corporation
2,Canadian corporate law,Canadian corporate lawconcerns the operation o...,Anguilla\nAustralia\nBVI\nCanada\nCayman Islan...,"['/wiki/Ontario', '/wiki/Series_LLC', '/wiki/C...",https://en.wikipedia.org/wiki/Canadian_corpora...
3,Australian corporate law,\nAustralian corporations lawhas historically ...,\nAnguilla\nAustralia\nBVI\nCanada\nCayman Isl...,"['/wiki/BBY_Limited', '/wiki/Series_LLC', '/wi...",https://en.wikipedia.org/wiki/Australian_corpo...
4,Indian Corporate Law Service,\nTheIndian Corporate Law Service(Hindi: भारती...,\nAmendment\nBasic structure doctrine\nFundame...,"['/wiki/District_Panchayat', '/wiki/List_of_st...",https://en.wikipedia.org/wiki/Indian_Corporate...


In [5]:
# to check for missing values in each row on the columns
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")
    


title
False    9998
Name: count, dtype: int64

summary
False    9102
True      896
Name: count, dtype: int64

content
False    9107
True      891
Name: count, dtype: int64

links
False    9998
Name: count, dtype: int64

url
False    9998
Name: count, dtype: int64



In [6]:
# percentage of the missing values as per columns
summary_percentage = missing_values['summary'].value_counts() / df['summary'].size 
content_percentage = missing_values['content'].value_counts() / df['summary'].size 

print(content_percentage)
print(summary_percentage)


content
False    0.910882
True     0.089118
Name: count, dtype: float64
summary
False    0.910382
True     0.089618
Name: count, dtype: float64


In [7]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)


number of duplicate rows:  (1804, 5)


In [8]:
#removing the duplicates
df = df.drop_duplicates()


In [10]:
# check if the duplicates are gone
print("number of duplicate rows: ", df.duplicated())


number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
9993    False
9994    False
9995    False
9996    False
9997    False
Length: 9998, dtype: bool


In [11]:
# Drop rows where both columns 'content' and 'summary' are empty
df1=df
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

In [12]:
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    9107
Name: count, dtype: int64

summary
False    9102
True        5
Name: count, dtype: int64

content
False    9107
Name: count, dtype: int64

links
False    9107
Name: count, dtype: int64

url
False    9107
Name: count, dtype: int64



In [13]:
# after removing the duplicates 
# we see most of the raws missed data both in content and summary sections
# this being only 5 rows we can replace them with Not Available

# Function to extract first 25 words
def get_summary(text):
    words = text.split()
    return ' '.join(words[:25]) if len(words) > 25 else text

# Update only rows where 'summary' is NaN
df_cleaned.loc[df['summary'].isna(), 'summary'] = df_cleaned['content'].apply(get_summary)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\3408418457.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.loc[df['summary'].isna(), 'summary'] = df_cleaned['content'].apply(get_summary)


In [14]:
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    9107
Name: count, dtype: int64

summary
False    9107
Name: count, dtype: int64

content
False    9107
Name: count, dtype: int64

links
False    9107
Name: count, dtype: int64

url
False    9107
Name: count, dtype: int64



In [15]:
# as seen from the dataset above, we have to remove the \n from the summary and content column
# there are many \n in the content and the summary column
#this removes all the \n from the content table and summart
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\1772156908.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\1772156908.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)


In [18]:

# Function to detect language
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"  # Handle errors

# Detect language
df_cleaned['Language'] = df['content'].apply(detect_language)

# Filter only non-English rows
non_english_df = df_cleaned[df_cleaned['Language'] != 'en'].copy()  # Copy to avoid warnings

# Function to translate text
def translate_text(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

# Apply translation **only to non-English rows**
non_english_df['EnglishText'] = non_english_df['Language'].apply(translate_text)

# Merge back translated texts into original DataFrame
df.update(non_english_df)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\3384308052.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Language'] = df['content'].apply(detect_language)


In [19]:
# removing links from the summary and content columns
#remove urls
import re

def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

#This function removes punctuations
def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
df_cleaned['title'] = df_cleaned['title'].apply(lambda x: remove_url(x))

C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\1304460945.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\1304460945.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
C:\Users\Administrator\AppData\Local\Temp\ipykernel_19404\1304460945.py:14: SettingWithCopyWarning: 
A value is tr

In [20]:
df_cleaned = df_cleaned.drop(columns=['links'])


In [21]:
import wordninja

# Apply word splitting
df_cleaned['summary'] =df_cleaned['summary'].apply(lambda x: " ".join(wordninja.split(x)))
df_cleaned['content'] =df_cleaned['content'].apply(lambda x: " ".join(wordninja.split(x)))

In [23]:
df_cleaned = df_cleaned.drop(columns=['Language'])


In [24]:
df_cleaned.head()

,title,summary,content,url
0,Corporate law,Corporate law also known as company law or ent...,Anguilla Australia BVI Canada Cayman Islands I...,https://en.wikipedia.org/wiki/Corporate_law
2,Canadian corporate law,Canadian corporate law concerns the operation ...,Anguilla Australia BVI Canada Cayman Islands I...,https://en.wikipedia.org/wiki/Canadian_corpora...
3,Australian corporate law,Australian corporations law has historically b...,Anguilla Australia BVI Canada Cayman Islands I...,https://en.wikipedia.org/wiki/Australian_corpo...
4,Indian Corporate Law Service,The Indian Corporate Law Service Hindi abbrevi...,Amendment Basic structure doctrine Fundamental...,https://en.wikipedia.org/wiki/Indian_Corporate...
5,United Kingdom company law,United Kingdom company law regulates corporati...,United Kingdom company law regulates corporati...,https://en.wikipedia.org/wiki/United_Kingdom_c...


In [25]:
df_cleaned.to_csv('cleaned_corporate.csv')